In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 23:25:16 WARN Utils: Your hostname, MSI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/08 23:25:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 23:25:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.version

'4.1.1'

In [3]:
!wget -O yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 23:25:36--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.64, 54.230.248.73, 54.230.248.205, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  4.41MB/s    in 17s     

2026-03-08 23:25:53 (4.09 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [4]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df.repartition(4).write.parquet('yellow_2025_11/', mode='overwrite')

In [5]:
!ls -lh yellow_2025_11/*.parquet

-rwxrwxrwx 1 swag swag 25M Mar  8 23:52 yellow_2025_11/part-00000-241520f1-d79d-4e95-b7b5-026473f68c97-c000.snappy.parquet
-rwxrwxrwx 1 swag swag 25M Mar  8 23:52 yellow_2025_11/part-00001-241520f1-d79d-4e95-b7b5-026473f68c97-c000.snappy.parquet
-rwxrwxrwx 1 swag swag 25M Mar  8 23:52 yellow_2025_11/part-00002-241520f1-d79d-4e95-b7b5-026473f68c97-c000.snappy.parquet
-rwxrwxrwx 1 swag swag 25M Mar  8 23:52 yellow_2025_11/part-00003-241520f1-d79d-4e95-b7b5-026473f68c97-c000.snappy.parquet


In [6]:
from pyspark.sql.functions import col, to_date

df.filter(to_date(col('tpep_pickup_datetime')) == '2025-11-15').count()

162604

In [8]:
from pyspark.sql.functions import col, max, unix_timestamp

df.withColumn('trip_hours',
    (unix_timestamp(col('tpep_dropoff_datetime')) - unix_timestamp(col('tpep_pickup_datetime'))) / 3600
).select(max('trip_hours')).show()

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



In [9]:
!wget -O taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 23:56:37--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.205, 54.230.248.73, 54.230.248.64, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.205|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-03-08 23:56:38 (14.9 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [10]:
df_zones = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')
df_zones.createOrReplaceTempView('zones')
df.createOrReplaceTempView('trips')

In [11]:
spark.sql("""
    SELECT z.Zone, COUNT(*) as cnt
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY cnt ASC
    LIMIT 5
""").show(truncate=False)

[Stage 12:======================================>                  (8 + 4) / 12]

+---------------------------------------------+---+
|Zone                                         |cnt|
+---------------------------------------------+---+
|Governor's Island/Ellis Island/Liberty Island|1  |
|Eltingville/Annadale/Prince's Bay            |1  |
|Arden Heights                                |1  |
|Port Richmond                                |3  |
|Rikers Island                                |4  |
+---------------------------------------------+---+

